# Multiperiod Heat Pumps

**Learning outcome:** Apply multiperiod heat pumps through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 5 to 30 minutes  
**Optional extras:** hpr

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Does one heat-pump concept remain useful across all operating periods?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Define shared-design validation

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
import math

from OpenPinch import PinchProblem
from OpenPinch.contracts.hpr import HPRTargetingError

def summarize_hpr_target(label, target):
    details = target.hpr_details
    record = details.target_simulation_record
    design_vector = details.design_vector
    load = target.hpr_load
    return {
        "name": label,
        "status": "feasible",
        "backend": details.simulation_backend,
        "cycle": target.hpr_cycle,
        "selected_load": None if load is None else float(load.selected),
        "achieved_load": None if load is None else float(load.achieved),
        "objective": float(details.obj),
        "period_ids": (
            [] if details.period_ids is None else list(details.period_ids)
        ),
        "period_weights": (
            []
            if details.period_weights is None
            else [float(value) for value in details.period_weights]
        ),
        "design_vector": (
            [] if design_vector is None else [float(value) for value in design_vector]
        ),
        "loop_count": 0 if record is None else len(record.loops),
    }

def summarize_hpr_failure(error):
    return {
        "status": "typed infeasible",
        "reason": str(error),
        "diagnostics": error.diagnostics.model_dump(mode="json"),
    }

EXPECTED_PERIODS = {"turndown", "base", "peak"}
SHARED_OPTIONS = {
    "HPR_MULTIPERIOD_OPTIMIZATION_ENABLED": True,
}

def new_shared_problem():
    return PinchProblem(
        "crude_preheat_train_multiperiod.json",
        project_name="Crude HPR",
    )

def validate_shared_target(label, target):
    assert target.hpr_success
    details = target.hpr_details
    assert details.design_vector is not None
    assert details.period_ids is not None
    assert set(details.period_ids) == EXPECTED_PERIODS
    assert details.period_weights is not None
    assert len(details.period_weights) == len(details.period_ids)
    assert all(math.isfinite(float(value)) for value in details.design_vector)
    assert math.isfinite(float(details.obj))
    assert details.period_outputs is not None
    assert set(details.period_outputs) == EXPECTED_PERIODS
    period_success = {
        period_id: bool(output["success"])
        for period_id, output in details.period_outputs.items()
    }
    assert all(period_success.values())
    assert details.weighted_output is not None
    weighted_objective = float(details.weighted_output["obj"])
    assert math.isfinite(weighted_objective)
    summary = summarize_hpr_target(label, target)
    summary["period_success"] = period_success
    summary["weighted_objective"] = weighted_objective
    return summary

## Step 2: Optimize the two Carnot designs

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
carnot_heat_pump_problem = new_shared_problem()
carnot_heat_pump = carnot_heat_pump_problem.target.carnot_heat_pump(
    period_id="base",
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
    options=SHARED_OPTIONS,
)
carnot_heat_pump_summary = validate_shared_target(
    "Carnot heat pump", carnot_heat_pump
)

carnot_refrigeration_problem = new_shared_problem()
carnot_refrigeration = (
    carnot_refrigeration_problem.target.carnot_refrigeration(
        period_id="base",
        load_fraction=0.25,
        condensers=1,
        evaporators=1,
        maximum_restarts=1,
        maximum_iterations=20,
        maximum_evaluations=50,
        options=SHARED_OPTIONS,
    )
)
carnot_refrigeration_summary = validate_shared_target(
    "Carnot refrigeration", carnot_refrigeration
)

## Step 3: Optimize three CoolProp-backed designs

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
vc_heat_pump_problem = new_shared_problem()
vc_heat_pump = vc_heat_pump_problem.target.vapour_compression_heat_pump(
    period_id="base",
    refrigerants=["water"],
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
    options=SHARED_OPTIONS,
)
vc_heat_pump_summary = validate_shared_target(
    "CoolProp VC heat pump", vc_heat_pump
)

vc_refrigeration_problem = new_shared_problem()
vc_refrigeration = (
    vc_refrigeration_problem.target.vapour_compression_refrigeration(
        period_id="base",
        refrigerants=["ammonia"],
        load_fraction=0.25,
        condensers=1,
        evaporators=1,
        maximum_restarts=1,
        maximum_iterations=20,
        maximum_evaluations=50,
        options=SHARED_OPTIONS,
    )
)
vc_refrigeration_summary = validate_shared_target(
    "CoolProp VC refrigeration", vc_refrigeration
)

mvr_problem = new_shared_problem()
mvr_heat_pump = mvr_problem.target.mvr_heat_pump(
    period_id="base",
    mvr_fluids=["Water"],
    mvr_stages=1,
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
    options={
        **SHARED_OPTIONS,
        "HPR_REFRIGERANTS": ["water"],
    },
)
mvr_summary = validate_shared_target(
    "CoolProp VC+MVR heat pump", mvr_heat_pump
)

shared_design_summaries = {
    "Carnot heat pump": carnot_heat_pump_summary,
    "Carnot refrigeration": carnot_refrigeration_summary,
    "CoolProp VC heat pump": vc_heat_pump_summary,
    "CoolProp VC refrigeration": vc_refrigeration_summary,
    "CoolProp VC+MVR heat pump": mvr_summary,
}

## Step 4: Screen one optional advanced cascade

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
# This is a sixth, separate optimization with a deliberately more
# complex topology and a tighter budget than the required screens.
advanced_problem = new_shared_problem()
try:
    advanced_cascade = (
        advanced_problem.target.vapour_compression_heat_pump(
            period_id="base",
            refrigerants=["water", "ammonia"],
            load_fraction=0.25,
            condensers=3,
            evaporators=2,
            maximum_restarts=1,
            maximum_iterations=5,
            maximum_evaluations=20,
            options=SHARED_OPTIONS,
        )
    )
    advanced_cascade_summary = validate_shared_target(
        "optional advanced CoolProp cascade", advanced_cascade
    )
except HPRTargetingError as error:
    advanced_cascade_summary = summarize_hpr_failure(error)

## Review the result

Compare the five independent shared-design optimizations through their common period, weight, design-vector, objective, and success evidence. The optional advanced cascade is a sixth and deliberately tighter screen, not a fallback for a required result.

In [ ]:
from IPython.display import display

display(shared_design_summaries)
display(advanced_cascade_summary)

## Interpret the result

A shared-design result is one installed design optimized across the complete period set. Inspect every period's feasibility, the aligned weights, the finite design vector, and the weighted objective before comparing technologies.

## Adapt this template

Run each candidate technology as a separate shared-design optimization. Use period-specific loads when plant availability or heat-source duty changes materially, and keep more complex cascade screens optional until a bounded one-stage design is credible.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.